In [0]:
%sql
CREATE CONNECTION IF NOT EXISTS my_http_conn
TYPE HTTP
OPTIONS (
  host 'https://earthquake.usgs.gov',
  port '443',
  base_path '/earthquakes/feed/v1.0/',
  bearer_token 'na'
);

In [0]:
dbutils.widgets.text('catalog_name','saiakhila_dev')
catalog_name = dbutils.widgets.get('catalog_name')


In [0]:
%py
spark.sql(f"use catalog {catalog_name}");
spark.sql("use schema bronze;")
spark.sql("create volume if not exists earthquake_data;")


In [0]:
from databricks.sdk import WorkspaceClient
import requests
import json
import datetime

w = WorkspaceClient()
conn = w.connections.get("my_http_conn")
base_url = f"{conn.options['host']}{conn.options.get('base_path', '')}".rstrip('/')  # Ensure base_url is defined and no trailing slash
url = f"{base_url}/summary/all_day.geojson"
response = requests.get(url)
if (response.status_code!= 200):
    raise Exception(f"Error in getting data from {url}")
data = response.json()
Current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(f'/Volumes/{catalog_name}/bronze/earthquake_data/data.json_{Current_date}',json.dumps(data),overwrite = True )